In [77]:
import requests
from config import APIConfig
import os
from loguru import logger
from datetime import datetime

In [84]:
class APIExtractor:

    def __init__(self,config # <- pass in APIConfig object
                 ,base_dir):

        self.config = config
        self.base_dir = base_dir
    
    def _build_request_params(self,query):

        headers = self.config.get_headers()
        params = {'$query':query}
        
        return self.config.base_url,params,headers
    
    def _get_raw_data_path(self,subset,date):

        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        directory_path = os.path.join(self.base_dir, 'raw_data', subset)
        os.makedirs(directory_path, exist_ok=True)
        return os.path.join(directory_path, f"{date}_{subset}_{timestamp}.csv")
    
    def _cleanup_old_files(self,directory_path,date):

        if not os.path.exists(directory_path):
            return

        try: 
            for filename in os.listdir(directory_path):
                if str(date) in filename:
                    old_file = os.path.join(directory_path, filename)
                    if os.path.exists(old_file):
                        os.remove(old_file)
                        logger.info(f"Removed old file: {old_file}")
                        break
        except FileNotFoundError:
            logger.error(f"Error: File '{old_file}' not found.")
    
    def save_to_csv(self, df, subset, date):

        directory_path = os.path.join(self.base_dir, 'raw_data', subset)
        self._cleanup_old_files(directory_path, date)
        
        try:
            file_path = self._get_raw_data_path(subset, date)
            df.to_csv(file_path, index=False)
            logger.info(f"Saved {len(df)} rows to {file_path}")
        except Exception as e:
            logger.error(f"An unexpected error occured!:{e}")
    
    def extract_data(self,url,subset,date):

        self.config.base_url = url
        offset = 0
        query = f"SELECT * WHERE crash_date='{date}' LIMIT 1000 OFFSET {offset}"
        logger.info(f'{query}')
        base_url, params, headers = self._build_request_params(query)
        
        try:
            response = requests.get(base_url, params=params, headers=headers)
            response.raise_for_status()
            
            logger.info(f"Extracting data for {subset} on {date}")
            temp_df = pd.json_normalize(response.json())
            df = temp_df.copy()

            while len(temp_df)==1000:
                offset+=1000
                offset_query = f"SELECT * WHERE crash_date='{date}' LIMIT 1000 OFFSET {offset}"
                base_url, params, headers = self._build_request_params(offset_query)
                offset_response = requests.get(base_url, params=params, headers=headers)
                temp_df = pd.json_normalize(offset_response.json())
                df = pd.concat([df, temp_df], axis=0, ignore_index=True)
            
            self.save_to_csv(df, subset, date)
            return df
            
        except requests.exceptions.RequestException as e:
            logger.error(f"Error extracting data: {e}")
            return Nones
    
    def get_latest_date(self, url):

        self.config.base_url = url
        query = "SELECT * ORDER BY crash_date DESC LIMIT 1"
        base_url, params, headers = self._build_request_params(query)
        
        try:
            response = requests.get(base_url, params=params, headers=headers)
            response.raise_for_status()
            
            df = pd.json_normalize(response.json())
            return pd.to_datetime(df['crash_date']).dt.date[0]
            
        except Exception as e:
            logger.error(f"Error getting latest date: {e}")
            return None

In [85]:
crash_api = APIConfig()
crash_extractor = APIExtractor(crash_api,'.')

In [86]:
crash_extractor.extract_data("https://data.cityofnewyork.us/resource/f55k-p6yu.json","persons",'2025-09-19')

2025-10-09 21:26:42.795 | INFO     | __main__:extract_data:56 - SELECT * WHERE crash_date='2025-09-19' LIMIT 1000 OFFSET 0
2025-10-09 21:26:45.085 | INFO     | __main__:extract_data:63 - Extracting data for persons on 2025-09-19
2025-10-09 21:26:46.594 | INFO     | __main__:_cleanup_old_files:34 - Removed old file: ./raw_data/persons/2025-09-19_persons_2025-10-09_21-24-10.csv
2025-10-09 21:26:46.624 | INFO     | __main__:save_to_csv:47 - Saved 1134 rows to ./raw_data/persons/2025-09-19_persons_2025-10-09_21-26-46.csv


,unique_id,collision_id,crash_date,crash_time,person_id,person_type,person_injury,vehicle_id,person_age,ped_role,...,ejection,emotional_status,bodily_injury,position_in_vehicle,safety_equipment,complaint,ped_location,ped_action,contributing_factor_1,contributing_factor_2
0,13481434,4843466,2025-09-19T00:00:00.000,17:55,ff04ad11-c4bf-4a4a-b6bf-3060ff6d55d3,Occupant,Unspecified,20939469,37,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13484275,4844156,2025-09-19T00:00:00.000,15:20,e47fc6fb-2cb0-49d6-bb1a-f06cf39ef262,Occupant,Unspecified,20941093,37,Passenger,...,Not Ejected,Does Not Apply,Does Not Apply,"Front passenger, if two or more persons, inclu...",Lap Belt & Harness,Does Not Apply,NaN,NaN,NaN,NaN
2,13482883,4844278,2025-09-19T00:00:00.000,8:02,9faf024b-cd93-4927-af60-9519f5e07eb3,Occupant,Unspecified,20940286,37,Driver,...,Not Ejected,Does Not Apply,Does Not Apply,Driver,Lap Belt & Harness,Does Not Apply,NaN,NaN,NaN,NaN
3,13483419,4843806,2025-09-19T00:00:00.000,1:40,ab7bbab1-1c10-417a-9b51-ad3c4f9d43dd,Occupant,Unspecified,20940572,25,Driver,...,Not Ejected,Does Not Apply,Does Not Apply,Driver,Unknown,Does Not Apply,NaN,NaN,NaN,NaN
4,13483048,4844242,2025-09-19T00:00:00.000,21:30,90bf1023-c43f-46f8-b687-12d25b590095,Occupant,Unspecified,20940372,23,Driver,...,Not Ejected,Does Not Apply,Does Not Apply,Driver,Unknown,Does Not Apply,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,13498829,4844237,2025-09-19T00:00:00.000,14:25,9761fe46-3820-40bf-ad21-d5d427e2f981,Occupant,Unspecified,20949413,11,Passenger,...,Not Ejected,Does Not Apply,Does Not Apply,"Front passenger, if two or more persons, inclu...",Unknown,Does Not Apply,NaN,NaN,NaN,NaN
1130,13499439,4848158,2025-09-19T00:00:00.000,23:15,863eb552-0846-4d59-8990-6eb000790a12,Occupant,Unspecified,20949780,82,Passenger,...,Not Ejected,Does Not Apply,Does Not Apply,Right rear passenger or motorcycle sidecar pas...,Lap Belt,Does Not Apply,NaN,NaN,NaN,NaN
1131,13499438,4848158,2025-09-19T00:00:00.000,23:15,925947dd-e919-423f-a31e-44caafa82450,Occupant,Unspecified,20949779,32,Driver,...,Not Ejected,Does Not Apply,Does Not Apply,Driver,Lap Belt,Does Not Apply,NaN,NaN,NaN,NaN
1132,13499326,4848168,2025-09-19T00:00:00.000,5:40,76dbfc1e-7c9c-4b8d-b3d7-e47cf0d1cf3e,Occupant,Unspecified,NaN,NaN,Witness,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [89]:
for sub in os.listdir('./raw_data'):
    for file in os.listdir(f'./raw_data/{sub}'):
        df = pd.read_csv(f'./raw_data/{sub}/{file}')
        if len(df)>=1000:
            print(file, len(df))

2025-09-19_persons_2025-10-09_21-32-50.csv 1134
2025-09-26_persons_2025-10-09_21-32-21.csv 1011
